# LangGraph vs Claude Agent SDK: Same Workload, Two Architectures

This executable Day 1 comparison uses the repository's insurance data to run the **same six model calls** through two orchestration styles:

- five independent specialists run in parallel: FNOL, claim, policy, duplicate, and control review;
- one synthesis call returns the shared `TriageDecision` contract;
- both paths use the same Anthropic model, evidence bundle, specialist prompts, synthesis prompt, and validation checks.

<img src="Images/d1pm_langgraph_vs_claude_sdk_architecture.svg" width="1100" alt="LangGraph declarative fan-out and fan-in compared with Claude Agent SDK imperative asynchronous orchestration">

> Run this notebook from top to bottom after the Day 1 AM and PM script chains. It makes 12 model calls for one complete comparison. Timing varies with network and provider load; one run is a teaching observation, not a statistically reliable benchmark.

## 1. What this comparison does—and does not prove

This is an **architecture lab**, not a vendor speed contest. LangGraph and Claude Agent SDK solve overlapping but different problems:

- **LangGraph** expresses orchestration as a compiled graph. Nodes return state updates, reducers merge parallel results, and a join edge makes synthesis wait for all specialists.
- **Claude Agent SDK** exposes an agent runtime. Python explicitly creates five independent `query()` streams with `asyncio.gather()`, collects their `ResultMessage` objects, then starts a synthesis query.

### Fairness controls

1. One evidence bundle is loaded once from `fnol_emails.csv` and `claims.db`.
2. Both paths use the same `ANTHROPIC_MODEL`, system instruction, task prompts, and JSON Schema.
3. Each path performs exactly five parallel specialist calls plus one synthesis call.
4. Neither path uses tools during the timed region; this isolates orchestration overhead from retrieval/tool variance.
5. We report wall time and provider-reported tokens. SDK process/session startup is part of the SDK measurement.

For a defensible performance conclusion, rerun both paths in alternating order across many cases and compare medians and percentiles. The single run here is designed for explanation and inspection.

In [2]:
print()

In [3]:
from __future__ import annotations

import asyncio
import csv
import json
import operator
import os
import sqlite3
import sys
from collections.abc import Callable, Coroutine
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from time import perf_counter
from typing import Annotated, Any, Literal, TypedDict, TypeVar

from claude_agent_sdk import ClaudeAgentOptions, ResultMessage, query
from dotenv import load_dotenv
from IPython.display import Markdown, display
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel, Field


def find_week2_root() -> Path:
    """Find Week2 whether Jupyter started in the repo, Week2, or this session folder."""
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd / "Week2", *cwd.parents]
    for candidate in candidates:
        if (candidate / "data" / "insurance" / "fnol_emails.csv").exists():
            return candidate
    raise FileNotFoundError("Start Jupyter from the repository or Week2 directory.")


ROOT = find_week2_root()
SESSION_DIR = ROOT / "W2D1" / "W2D1PM"
FNOL_PATH = ROOT / "data" / "insurance" / "fnol_emails.csv"
CLAIMS_DB = ROOT / "data" / "insurance" / "claims.db"
load_dotenv(ROOT / ".env")

T = TypeVar("T")


def run_sdk(factory: Callable[[], Coroutine[Any, Any, T]]) -> T:
    """Run Claude Agent SDK work on a loop that can spawn Claude Code.

    On Windows, Jupyter often uses SelectorEventLoop, which cannot create
    subprocesses. The SDK needs ProactorEventLoop for the Claude Code CLI.
    """

    def _runner() -> T:
        loop = (
            asyncio.ProactorEventLoop()
            if sys.platform == "win32"
            else asyncio.new_event_loop()
        )
        asyncio.set_event_loop(loop)
        try:
            return loop.run_until_complete(factory())
        finally:
            loop.run_until_complete(loop.shutdown_asyncgens())
            loop.close()

    with ThreadPoolExecutor(max_workers=1) as pool:
        return pool.submit(_runner).result()


if not os.getenv("ANTHROPIC_API_KEY"):
    raise RuntimeError("Set ANTHROPIC_API_KEY in Week2/.env before running this notebook.")
if not CLAIMS_DB.exists():
    raise RuntimeError("Run: uv run python data/insurance/seed_claims_db.py")

MODEL_NAME = os.getenv("ANTHROPIC_MODEL", "claude-sonnet-4-6")
MAX_TOKENS = int(os.getenv("AGENT_MAX_TOKENS", "8000"))
MAX_TURNS = int(os.getenv("AGENT_MAX_STEPS", "10"))
MAX_BUDGET_USD = float(os.getenv("AGENT_MAX_BUDGET_USD", "0.50"))
TIMEOUT_SECONDS = int(os.getenv("AGENT_REQUEST_TIMEOUT_SECONDS", "60"))

print(f"Week2 root: {ROOT}")
print(f"Model: {MODEL_NAME}")

c:\Users\Asus\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Week2 root: D:\Ambilio\EXL-CampusHire\Week2
Model: claude-sonnet-4-6


## 2. Shared evidence and output contract

Functions introduced here:

- `BaseModel` / `Field`: define and validate the common `TriageDecision` contract.
- `model_json_schema()`: gives the Claude Agent SDK the same schema used by LangChain structured output.
- `model_validate()`: validates SDK `structured_output`; no display-text JSON parsing.
- `sqlite3.Row`: converts authoritative database rows to dictionaries without inventing fields.

The loader runs **before** either timer starts. Both architectures therefore receive byte-for-byte equivalent JSON evidence.

In [4]:
class TriageDecision(BaseModel):
    """Stable decision contract shared by both architectures."""

    lane: Literal["insurance"]
    case_id: str = Field(pattern=r"^CLM-\d{6}$")
    urgency: Literal["low", "medium", "high"]
    route_queue: str
    summary: str
    rationale: str
    tools_used: list[str]
    confidence: float = Field(ge=0.0, le=1.0)
    needs_human_approval: bool = False


class RunMetric(TypedDict):
    operation: str
    seconds: float
    model_calls: int
    input_tokens: int
    output_tokens: int
    cache_creation_tokens: int
    cache_read_tokens: int
    total_tokens: int


SPECIALISTS = {
    "fnol": "Assess the incoming notice, incident facts, and urgency signals.",
    "claim": "Check the authoritative claim record and identify status or consistency issues.",
    "policy": "Check the authoritative policy record and identify coverage/status signals.",
    "duplicate": "Compare related notices and determine whether this is a duplicate or follow-up.",
    "control": "Check expected route, urgency, identifiers, and whether human approval is required.",
}

SYSTEM_PROMPT = (
    "You are a concise insurance operations specialist. Use only supplied evidence. "
    "Never invent a claim, policy, person, amount, or status. State 'not found' when evidence "
    "is absent. Return no more than 120 words and cite exact evidence identifiers."
)

SYNTHESIS_SYSTEM_PROMPT = (
    "You synthesize grounded insurance specialist reports into TriageDecision. Use only the "
    "supplied reports and evidence. case_id must be the CLM claim identifier, not the CLN email "
    "identifier. tools_used must list all five specialist names. Never authorize a write."
)


def load_case_bundle(email_id: str = "CLN-001") -> dict[str, Any]:
    """Load one case and all benchmark evidence from repository fixtures."""
    with FNOL_PATH.open(encoding="utf-8", newline="") as handle:
        notices = list(csv.DictReader(handle))
    fnol = next((row for row in notices if row["email_id"] == email_id), None)
    if fnol is None:
        raise ValueError(f"FNOL record {email_id} was not found.")

    claim_id = fnol["claim_number_ground_truth"]
    policy_id = fnol["policy_id_ground_truth"]
    related = [
        row for row in notices
        if row["email_id"] != email_id and row["claim_number_ground_truth"] == claim_id
    ]
    with sqlite3.connect(CLAIMS_DB) as connection:
        connection.row_factory = sqlite3.Row
        claim_row = connection.execute(
            "SELECT * FROM claims WHERE claim_id = ?", (claim_id,)
        ).fetchone()
        policy_row = connection.execute(
            "SELECT * FROM policies WHERE policy_id = ?", (policy_id,)
        ).fetchone()

    return {
        "fnol": fnol,
        "claim": dict(claim_row) if claim_row else {"found": False, "claim_id": claim_id},
        "policy": dict(policy_row) if policy_row else {"found": False, "policy_id": policy_id},
        "duplicate": {"source_email_id": email_id, "related_notices": related},
        "control": {
            "expected_claim_id": claim_id,
            "expected_urgency": fnol["urgency_ground_truth"],
            "expected_route_queue": fnol["route_queue_ground_truth"],
            "write_allowed": False,
        },
    }


def specialist_prompt(name: str, evidence: Any) -> str:
    return (
        f"Operation: {name}\nResponsibility: {SPECIALISTS[name]}\n"
        f"Evidence:\n{json.dumps(evidence, sort_keys=True, default=str)}"
    )


def synthesis_prompt(reports: list[dict[str, str]]) -> str:
    return (
        "Create the final grounded TriageDecision from these five reports:\n"
        + json.dumps(sorted(reports, key=lambda item: item["operation"]), sort_keys=True)
    )


def normalize_usage(
    operation: str,
    seconds: float,
    usage: dict[str, Any] | None,
    model_calls: int = 1,
) -> RunMetric:
    usage = usage or {}
    details = usage.get("input_token_details", {}) or {}
    raw_input = int(usage.get("input_tokens", 0) or 0)
    output_tokens = int(usage.get("output_tokens", 0) or 0)
    cache_creation = int(
        usage.get("cache_creation_input_tokens", details.get("cache_creation", 0)) or 0
    )
    cache_read = int(
        usage.get("cache_read_input_tokens", details.get("cache_read", 0)) or 0
    )
    # Native Anthropic/SDK usage reports non-cached input separately; LangChain's
    # standard usage_metadata input_tokens already includes cached input.
    sdk_style_usage = (
        "cache_creation_input_tokens" in usage or "cache_read_input_tokens" in usage
    )
    effective_input = raw_input + cache_creation + cache_read if sdk_style_usage else raw_input
    return {
        "operation": operation,
        "seconds": round(seconds, 3),
        "model_calls": model_calls,
        "input_tokens": effective_input,
        "output_tokens": output_tokens,
        "cache_creation_tokens": cache_creation,
        "cache_read_tokens": cache_read,
        "total_tokens": effective_input + output_tokens,
    }


CASE_BUNDLE = load_case_bundle("CLN-001")
print(json.dumps(CASE_BUNDLE, indent=2, default=str))

{
  "fnol": {
    "email_id": "CLN-001",
    "subject": "Property claim CLM-424063 \u2014 fence damage",
    "body": "Jennifer Garcia reports vandalism at 6650 Broadway. Claim CLM-424063. Policy POL-787532354. Police report PR-20250822-4864. Estimated damage $5,834. Photos available.",
    "email_category": "fnol",
    "urgency_ground_truth": "high",
    "claim_number_ground_truth": "CLM-424063",
    "route_queue_ground_truth": "fnol_intake",
    "policy_id_ground_truth": "POL-787532354"
  },
  "claim": {
    "claim_id": "CLM-424063",
    "policy_id": "POL-787532354",
    "status": "open",
    "loss_type": "property",
    "urgency": "high",
    "reserve_usd": 5834.0,
    "route_queue": "fnol_intake"
  },
  "policy": {
    "policy_id": "POL-787532354",
    "named_insured": "Jennifer Garcia",
    "status": "active",
    "product": "homeowners"
  },
  "duplicate": {
    "source_email_id": "CLN-001",
    "related_notices": [
      {
        "email_id": "CLN-009",
        "subject": "Duplic

## 3. Architecture A—LangGraph

Functions and types introduced here:

- `StateGraph(ComparisonState)`: creates a graph whose node input/output follows typed shared state.
- `Annotated[..., operator.add]`: reducer metadata; parallel nodes append reports and metrics instead of overwriting one another.
- `add_node()`: registers each specialist and the synthesis function.
- `add_edge(START, node)`: creates fan-out from the virtual start node.
- `add_edge(list_of_nodes, "synthesize")`: creates a fan-in join; synthesis waits for every specialist.
- `compile()`: validates topology and produces the runnable graph.
- `ainvoke()`: executes the graph asynchronously.
- `with_structured_output(..., include_raw=True)`: validates the decision while retaining the raw message needed for token accounting.

The graph owns dependency scheduling and state merging. Application code declares topology; the runtime determines which ready nodes can execute concurrently.

In [5]:
class ComparisonState(TypedDict):
    evidence: dict[str, Any]
    reports: Annotated[list[dict[str, str]], operator.add]
    metrics: Annotated[list[RunMetric], operator.add]
    final_decision: dict[str, Any]


langgraph_model = ChatAnthropic(
    model=MODEL_NAME,
    max_tokens=MAX_TOKENS,
    temperature=0,
)
langgraph_synthesizer = langgraph_model.with_structured_output(
    TriageDecision,
    include_raw=True,
)


def message_text(message: Any) -> str:
    """Normalize LangChain string or content-block responses."""
    if isinstance(message.content, str):
        return message.content
    return "\n".join(
        str(block.get("text", "")) if isinstance(block, dict) else str(block)
        for block in message.content
        if not isinstance(block, dict) or block.get("type") == "text"
    )


def make_langgraph_specialist(name: str):
    async def specialist(state: ComparisonState) -> dict[str, Any]:
        started = perf_counter()
        response = await langgraph_model.ainvoke(
            [
                SystemMessage(content=SYSTEM_PROMPT),
                HumanMessage(content=specialist_prompt(name, state["evidence"][name])),
            ]
        )
        elapsed = perf_counter() - started
        return {
            "reports": [{"operation": name, "report": message_text(response)}],
            "metrics": [normalize_usage(name, elapsed, response.usage_metadata)],
        }

    return specialist


async def langgraph_synthesize(state: ComparisonState) -> dict[str, Any]:
    started = perf_counter()
    response = await langgraph_synthesizer.ainvoke(
        [
            SystemMessage(content=SYNTHESIS_SYSTEM_PROMPT),
            HumanMessage(content=synthesis_prompt(state["reports"])),
        ]
    )
    elapsed = perf_counter() - started
    if response["parsed"] is None:
        raise RuntimeError(f"LangGraph synthesis schema error: {response.get('parsing_error')}")
    decision = TriageDecision.model_validate(response["parsed"])
    raw_message = response["raw"]
    return {
        "final_decision": decision.model_dump(),
        "metrics": [normalize_usage("synthesis", elapsed, raw_message.usage_metadata)],
    }


langgraph_builder = StateGraph(ComparisonState)
langgraph_node_names = []
for specialist_name in SPECIALISTS:
    node_name = f"review_{specialist_name}"
    langgraph_node_names.append(node_name)
    langgraph_builder.add_node(node_name, make_langgraph_specialist(specialist_name))
    langgraph_builder.add_edge(START, node_name)

langgraph_builder.add_node("synthesize", langgraph_synthesize)
langgraph_builder.add_edge(langgraph_node_names, "synthesize")
langgraph_builder.add_edge("synthesize", END)
langgraph_app = langgraph_builder.compile()

print("Compiled nodes:", list(langgraph_app.get_graph().nodes))

Compiled nodes: ['__start__', 'review_fnol', 'review_claim', 'review_policy', 'review_duplicate', 'review_control', 'synthesize', '__end__']


In [6]:
langgraph_started = perf_counter()
langgraph_result = await asyncio.wait_for(
    langgraph_app.ainvoke(
        {
            "evidence": CASE_BUNDLE,
            "reports": [],
            "metrics": [],
            "final_decision": {},
        },
        config={"recursion_limit": MAX_TURNS},
    ),
    timeout=TIMEOUT_SECONDS * 2,
)
langgraph_wall_seconds = round(perf_counter() - langgraph_started, 3)

print(f"LangGraph wall time: {langgraph_wall_seconds}s")
print(json.dumps(langgraph_result["final_decision"], indent=2))
display(Markdown("### LangGraph specialist reports"))
for report in sorted(langgraph_result["reports"], key=lambda item: item["operation"]):
    display(Markdown(f"**{report['operation']}** — {report['report']}"))

LangGraph wall time: 18.74s
{
  "lane": "insurance",
  "case_id": "CLM-424063",
  "urgency": "high",
  "route_queue": "fnol_intake",
  "summary": "Claimant Jennifer Garcia (Policy POL-787532354, active Homeowners) filed a vandalism claim (CLM-424063) for fence damage at 6650 Broadway, with an estimated loss of $5,834. A police report (PR-20250822-4864) has been filed and photographic evidence is available. A follow-up communication (CLN-009) adds actionable scheduling information \u2014 a contractor is ready to begin Monday pending approval \u2014 and should be attached to this claim rather than treated as a duplicate. A reserve of $5,834 has been set, though this is inconsistent with the claim still sitting in fnol_intake. Automated writes are blocked; human approval is required before any write/update actions are taken.",
  "rationale": "1. FNOL Specialist confirmed high urgency based on an active vandalism claim with a police report, documented damage estimate ($5,834), and photogra

### LangGraph specialist reports

**claim** — ## Claim Review: CLM-424063

**Status:** Open | **Urgency:** High

### Consistency Issues Identified:

1. **Queue Mismatch:** `route_queue` is set to **"fnol_intake"**, but the claim status is **"open"** — an open claim should typically have progressed beyond FNOL intake to an active handling queue. This suggests a routing or workflow progression issue.

2. **Reserve Set:** A reserve of **$5,834.00 USD** is established, which is inconsistent with a claim still sitting in `fnol_intake` — reserves are generally assigned post-triage/assignment.

### Recommended Actions:
- Verify whether claim was properly advanced from FNOL to active adjuster queue.
- Confirm reserve authorization aligns with current workflow stage.

**Evidence Source:** CLM-424063 record fields: `status`, `route_queue`, `reserve_usd`.

**control** — **Control Check Results**

| Field | Expected | Status |
|---|---|---|
| Claim ID | CLM-424063 | ✅ Confirmed |
| Route Queue | fnol_intake | ✅ Confirmed |
| Urgency | High | ✅ Confirmed |
| Write Allowed | False | ⚠️ **Write BLOCKED** |

**Human Approval Required:** Yes — `write_allowed: false` per evidence record prohibits automated write operations.

**Action:** Route CLM-424063 to `fnol_intake` queue at high urgency, but **halt before any write/update actions** pending human approval.

*Source: Evidence identifiers — expected_claim_id, expected_route_queue, expected_urgency, write_allowed.*

**duplicate** — ## Duplicate/Follow-Up Analysis

**Source Email:** CLN-001
**Related Notice:** CLN-009

**Determination: Follow-Up (not a pure duplicate)**

CLN-009's body explicitly states *"Following up on CLM-424063 — contractor can start Monday if approved,"* indicating it is a **follow-up communication** providing new actionable information (contractor scheduling) rather than an identical re-submission of the original notice.

**Key Evidence:**
- CLN-009 subject: "Duplicate notice CLM-424063" — labeled duplicate, but body content adds new information
- CLN-009 body references approval pending and a Monday start date
- Claim: CLM-424063 | Policy: POL-787532354 | Queue: fnol_intake | Urgency: medium

**Recommendation:** Treat as follow-up; attach to CLM-424063 rather than creating a new claim.

**fnol** — ## FNOL Assessment — CLN-001

**Claimant:** Jennifer Garcia
**Claim:** CLM-424063
**Policy:** POL-787532354
**Incident:** Vandalism — fence damage at 6650 Broadway
**Estimated Damage:** $5,834
**Police Report:** PR-20250822-4864
**Supporting Evidence:** Photos available

**Urgency: HIGH** — Active vandalism claim with police report filed, documented damage estimate, and photographic evidence ready for review.

**Recommended Queue:** `fnol_intake`

**Next Steps:** Acknowledge receipt, verify policy POL-787532354 coverage, retrieve police report PR-20250822-4864, and collect available photos for damage validation.

*All findings sourced from evidence identifier CLN-001.*

**policy** — **Policy Check Results**

- **Named Insured:** Jennifer Garcia
- **Policy ID:** POL-787532354
- **Product:** Homeowners
- **Status:** Active ✅

**Coverage/Status Signals:**
The policy record confirms an **active** homeowners policy for Jennifer Garcia under POL-787532354. No exclusions, endorsements, coverage limits, deductibles, or lapse indicators are present in the supplied evidence.

**Note:** Specific coverage details (dwelling limits, liability, deductibles, etc.) are **not found** in the provided evidence and should be retrieved from the full policy document before making coverage determinations.

*Source: Policy record POL-787532354*

## 4. Architecture B—Claude Agent SDK

Functions and types introduced here:

- `ClaudeAgentOptions`: configures model, system prompt, tools, turn/budget controls, working directory, and optional structured output.
- `query()`: starts one SDK agent session and returns an asynchronous stream of native messages.
- `ResultMessage`: authoritative terminal result containing text, structured output, session ID, turn count, usage, and estimated cost.
- `asyncio.gather()`: application-owned fan-out/fan-in for the five independent SDK sessions.
- `asyncio.wait_for()`: outer timeout around each complete message stream.

`tools=[]` and `setting_sources=[]` intentionally remove tool and project-context differences from the timed comparison. This does **not** mean the SDK lacks tools; Topic 2 already demonstrates in-process MCP tools. Here, Python owns orchestration while the SDK owns each model-backed agent session.

In [7]:
def sdk_options(system_prompt: str, *, structured: bool = False) -> ClaudeAgentOptions:
    """Build isolated SDK options for one benchmark operation."""
    values: dict[str, Any] = {
        "model": MODEL_NAME,
        "system_prompt": system_prompt,
        "tools": [],
        "max_turns": MAX_TURNS,
        "max_budget_usd": MAX_BUDGET_USD,
        "cwd": SESSION_DIR,
        "setting_sources": [],
    }
    if structured:
        values["output_format"] = {
            "type": "json_schema",
            "schema": TriageDecision.model_json_schema(),
        }
    return ClaudeAgentOptions(**values)


async def collect_sdk_result(prompt: str, options: ClaudeAgentOptions) -> ResultMessage:
    """Consume one native SDK stream and return its terminal result."""
    terminal: ResultMessage | None = None

    async def consume() -> None:
        nonlocal terminal
        async for message in query(prompt=prompt, options=options):
            if isinstance(message, ResultMessage):
                terminal = message

    await asyncio.wait_for(consume(), timeout=TIMEOUT_SECONDS)
    if terminal is None:
        raise RuntimeError("Claude Agent SDK stream ended without ResultMessage.")
    if terminal.is_error:
        raise RuntimeError(f"Claude Agent SDK failed: {terminal.subtype}")
    return terminal


async def run_sdk_specialist(name: str) -> dict[str, Any]:
    started = perf_counter()
    result = await collect_sdk_result(
        specialist_prompt(name, CASE_BUNDLE[name]),
        sdk_options(SYSTEM_PROMPT),
    )
    elapsed = perf_counter() - started
    return {
        "operation": name,
        "report": result.result or "",
        "metric": normalize_usage(name, elapsed, result.usage, result.num_turns),
        "session_id": result.session_id,
        "cost_usd": result.total_cost_usd,
    }


async def run_sdk_synthesis(reports: list[dict[str, str]]) -> dict[str, Any]:
    started = perf_counter()
    result = await collect_sdk_result(
        synthesis_prompt(reports),
        sdk_options(SYNTHESIS_SYSTEM_PROMPT, structured=True),
    )
    elapsed = perf_counter() - started
    decision = TriageDecision.model_validate(result.structured_output)
    return {
        "decision": decision.model_dump(),
        "metric": normalize_usage("synthesis", elapsed, result.usage, result.num_turns),
        "session_id": result.session_id,
        "cost_usd": result.total_cost_usd,
    }

In [8]:
sdk_started = perf_counter()


async def _sdk_benchmark() -> tuple[list[dict[str, Any]], dict[str, Any]]:
    sdk_specialist_runs = await asyncio.gather(
        *(run_sdk_specialist(name) for name in SPECIALISTS)
    )
    sdk_reports = [
        {"operation": run["operation"], "report": run["report"]}
        for run in sdk_specialist_runs
    ]
    sdk_synthesis = await run_sdk_synthesis(sdk_reports)
    return sdk_specialist_runs, sdk_synthesis


try:
    sdk_specialist_runs, sdk_synthesis = run_sdk(_sdk_benchmark)
except Exception as error:
    raise RuntimeError(
        "Agent SDK comparison failed. Check Anthropic credits, model access, "
        "credentials, and the Claude Code CLI runtime. Some SDK versions may report "
        "'error result: success' after an upstream credit error."
    ) from error

sdk_reports = [
    {"operation": run["operation"], "report": run["report"]}
    for run in sdk_specialist_runs
]
sdk_wall_seconds = round(perf_counter() - sdk_started, 3)
sdk_metrics = [run["metric"] for run in sdk_specialist_runs] + [sdk_synthesis["metric"]]
sdk_cost_usd = sum(float(run["cost_usd"] or 0) for run in sdk_specialist_runs) + float(
    sdk_synthesis["cost_usd"] or 0
)

print(f"Claude Agent SDK wall time: {sdk_wall_seconds}s")
print(f"SDK estimated cost: ${sdk_cost_usd:.6f}")
print(json.dumps(sdk_synthesis["decision"], indent=2))
display(Markdown("### Claude Agent SDK specialist reports"))
for report in sorted(sdk_reports, key=lambda item: item["operation"]):
    display(Markdown(f"**{report['operation']}** — {report['report']}"))

Claude Agent SDK wall time: 49.092s
SDK estimated cost: $0.068508
{
  "lane": "insurance",
  "case_id": "CLM-424063",
  "urgency": "high",
  "route_queue": "fnol_intake",
  "summary": "High-urgency property vandalism claim filed by Jennifer Garcia under active homeowners policy POL-787532354. Incident: fence damage at 6650 Broadway; estimated loss $5,834 (reserve already established). Police report PR-20250822-4864 filed; photographic evidence available. Follow-up email CLN-009 indicates a contractor is ready to begin work Monday pending approval \u2014 no new claim should be opened (CLN-001 is a duplicate). A workflow stall is present: the reserve has been set but the claim remains in fnol_intake with no adjuster assigned. Write operations are prohibited; human approval is required before any further action.",
  "rationale": "All five specialist reports converge on CLM-424063 as a single, open high-urgency property claim. (1) Claim specialist confirms the reserve of $5,834 is set and 

### Claude Agent SDK specialist reports

**claim** — **Claim Review: CLM-424063**

**Record Summary:**
- Policy: POL-787532354 | Loss: Property | Status: Open | Urgency: High
- Reserve: $5,834.00 | Queue: fnol_intake

**Consistency Issues Identified:**

1. **Queue/Reserve Mismatch:** A reserve of **$5,834.00** is already established, yet the claim remains in **fnol_intake**. Reserve-setting typically occurs post-triage; this suggests the claim has not advanced despite financial evaluation being completed.

2. **Urgency vs. Queue Lag:** Urgency is flagged **high**, but the claim sits in the initial intake queue — indicating a likely **routing failure or workflow stall**.

**Recommended Action:** Escalate CLM-424063 out of fnol_intake to active adjustment queue immediately.

*Source: CLM-424063 record (supplied evidence)*

**control** — **Control Check — Summary**

| Field | Value | Source |
|---|---|---|
| Claim ID | CLM-424063 | Evidence `expected_claim_id` |
| Route Queue | `fnol_intake` | Evidence `expected_route_queue` |
| Urgency | **High** | Evidence `expected_urgency` |
| Write Allowed | **No** | Evidence `write_allowed: false` |

**Human Approval Required: YES** — `write_allowed` is `false`; no write operations may proceed without explicit human authorization.

All four control fields are present and verified. No invented data. Routing to `fnol_intake` with high urgency is confirmed per supplied evidence.

**duplicate** — **Duplicate/Follow-Up Assessment — CLN-001 vs. CLN-009**

**Finding: Duplicate with follow-up content.**

Per evidence, related notice **CLN-009** has subject *"Duplicate notice CLM-424063"* and body *"Following up on CLM-424063 — contractor can start Monday if approved."* This confirms CLN-001 is tied to an already-existing claim.

| Field | Value |
|---|---|
| Source Email | CLN-001 |
| Related Email | CLN-009 |
| Claim Reference | CLM-424063 |
| Policy | POL-787532354 |
| Category | fnol |
| Queue | fnol_intake |
| Urgency | Medium |

**Recommendation:** Flag CLN-001 as duplicate of CLM-424063; route contractor approval note to `fnol_intake` queue for adjuster action. Do not open a new claim.

*Citations: CLN-009 subject, body, claim_number_ground_truth, policy_id_ground_truth.*

**fnol** — ## FNOL Assessment — CLM-424063

**Claimant:** Jennifer Garcia
**Policy:** POL-787532354
**Claim:** CLM-424063
**Incident:** Vandalism at 6650 Broadway (fence damage)
**Estimated Damage:** $5,834
**Police Report:** PR-20250822-4864
**Supporting Evidence:** Photos available

**Urgency: HIGH**
Property crime with police report filed and photographic evidence ready for review warrants expedited handling.

**Recommended Queue:** `fnol_intake`

**Next Steps:** Confirm policy coverage, retrieve photos, validate damage estimate against PR-20250822-4864, and assign adjuster.

*Sources: CLN-001 (email body, subject, ground-truth fields)*

**policy** — **Policy Check — POL-787532354**

| Field | Value |
|---|---|
| **Named Insured** | Jennifer Garcia |
| **Policy ID** | POL-787532354 |
| **Product** | Homeowners |
| **Status** | **Active** |

**Coverage/Status Signal:** Policy is currently **active** as of today (2026-07-20), confirming valid homeowners coverage for the named insured.

**Not found:** Coverage limits, deductibles, effective/expiration dates, endorsements, or exclusions — these details are absent from the supplied evidence.

*Source: Policy record POL-787532354*

## 5. Compare runtime, tokens, calls, and grounding

Interpret the measurements carefully:

- **Wall time** answers what the operator experienced end to end.
- **Summed operation time** is larger than wall time when specialists overlap; it is useful for visualizing concurrency, not elapsed time.
- **Input tokens** include cache-read/cache-creation input where provider metadata exposes it.
- **Model calls** use one call per LangGraph node and SDK `num_turns` per session.
- **Grounding checks** verify observable contract facts. They do not prove every sentence is correct.

The detailed operation table helps reveal a straggler that the aggregate totals would hide.

In [9]:
def aggregate_metrics(metrics: list[RunMetric], wall_seconds: float) -> dict[str, Any]:
    specialist_seconds = [m["seconds"] for m in metrics if m["operation"] != "synthesis"]
    return {
        "wall_seconds": wall_seconds,
        "slowest_specialist_seconds": max(specialist_seconds),
        "summed_operation_seconds": round(sum(m["seconds"] for m in metrics), 3),
        "model_calls": sum(m["model_calls"] for m in metrics),
        "input_tokens": sum(m["input_tokens"] for m in metrics),
        "output_tokens": sum(m["output_tokens"] for m in metrics),
        "total_tokens": sum(m["total_tokens"] for m in metrics),
    }


def grounding_checks(decision: dict[str, Any], evidence: dict[str, Any]) -> dict[str, bool]:
    expected = evidence["control"]
    return {
        "claim_id_matches": decision["case_id"] == expected["expected_claim_id"],
        "urgency_matches": decision["urgency"] == expected["expected_urgency"],
        "route_matches": decision["route_queue"] == expected["expected_route_queue"],
        "all_specialists_cited": set(decision["tools_used"]) == set(SPECIALISTS),
        "no_write_authorized": decision["needs_human_approval"] is False,
    }


langgraph_summary = aggregate_metrics(langgraph_result["metrics"], langgraph_wall_seconds)
sdk_summary = aggregate_metrics(sdk_metrics, sdk_wall_seconds)
langgraph_checks = grounding_checks(langgraph_result["final_decision"], CASE_BUNDLE)
sdk_checks = grounding_checks(sdk_synthesis["decision"], CASE_BUNDLE)

comparison_rows = [
    ("Wall time (s)", langgraph_summary["wall_seconds"], sdk_summary["wall_seconds"]),
    ("Slowest specialist (s)", langgraph_summary["slowest_specialist_seconds"], sdk_summary["slowest_specialist_seconds"]),
    ("Summed operation time (s)", langgraph_summary["summed_operation_seconds"], sdk_summary["summed_operation_seconds"]),
    ("Model calls / SDK turns", langgraph_summary["model_calls"], sdk_summary["model_calls"]),
    ("Input tokens", langgraph_summary["input_tokens"], sdk_summary["input_tokens"]),
    ("Output tokens", langgraph_summary["output_tokens"], sdk_summary["output_tokens"]),
    ("Total tokens", langgraph_summary["total_tokens"], sdk_summary["total_tokens"]),
    ("Grounding checks passed", f"{sum(langgraph_checks.values())}/5", f"{sum(sdk_checks.values())}/5"),
]

summary_table = [
    "| Metric | LangGraph | Claude Agent SDK |",
    "|---|---:|---:|",
    *[f"| {name} | {left} | {right} |" for name, left, right in comparison_rows],
]
display(Markdown("\n".join(summary_table)))

detail_rows = []
for architecture, metrics in (
    ("LangGraph", langgraph_result["metrics"]),
    ("Claude Agent SDK", sdk_metrics),
):
    for metric in sorted(metrics, key=lambda item: item["operation"]):
        detail_rows.append(
            f"| {architecture} | {metric['operation']} | {metric['seconds']} | "
            f"{metric['model_calls']} | {metric['input_tokens']} | "
            f"{metric['output_tokens']} | {metric['total_tokens']} |"
        )

detail_table = [
    "### Per-operation evidence",
    "| Architecture | Operation | Seconds | Calls/turns | Input | Output | Total |",
    "|---|---|---:|---:|---:|---:|---:|",
    *detail_rows,
]
display(Markdown("\n".join(detail_table)))
print("LangGraph grounding:", langgraph_checks)
print("Claude Agent SDK grounding:", sdk_checks)

| Metric | LangGraph | Claude Agent SDK |
|---|---:|---:|
| Wall time (s) | 18.74 | 49.092 |
| Slowest specialist (s) | 6.938 | 15.577 |
| Summed operation time (s) | 38.782 | 94.379 |
| Model calls / SDK turns | 6 | 7 |
| Input tokens | 2933 | 3688 |
| Output tokens | 1611 | 3369 |
| Total tokens | 4544 | 7057 |
| Grounding checks passed | 4/5 | 4/5 |

### Per-operation evidence
| Architecture | Operation | Seconds | Calls/turns | Input | Output | Total |
|---|---|---:|---:|---:|---:|---:|
| LangGraph | claim | 6.083 | 1 | 156 | 215 | 371 |
| LangGraph | control | 4.888 | 1 | 134 | 185 | 319 |
| LangGraph | duplicate | 6.938 | 1 | 226 | 214 | 440 |
| LangGraph | fnol | 4.753 | 1 | 242 | 186 | 428 |
| LangGraph | policy | 4.407 | 1 | 123 | 160 | 283 |
| LangGraph | synthesis | 11.713 | 1 | 2052 | 651 | 2703 |
| Claude Agent SDK | claim | 15.577 | 1 | 279 | 478 | 757 |
| Claude Agent SDK | control | 10.468 | 1 | 257 | 283 | 540 |
| Claude Agent SDK | duplicate | 13.896 | 1 | 349 | 483 | 832 |
| Claude Agent SDK | fnol | 10.62 | 1 | 365 | 216 | 581 |
| Claude Agent SDK | policy | 10.308 | 1 | 246 | 190 | 436 |
| Claude Agent SDK | synthesis | 33.51 | 2 | 2192 | 1719 | 3911 |

LangGraph grounding: {'claim_id_matches': True, 'urgency_matches': True, 'route_matches': True, 'all_specialists_cited': True, 'no_write_authorized': False}
Claude Agent SDK grounding: {'claim_id_matches': True, 'urgency_matches': True, 'route_matches': True, 'all_specialists_cited': True, 'no_write_authorized': False}


## 6. Architecture decision guide

Choose **LangGraph** when the workflow benefits from explicit state transitions, conditional edges, durable checkpoints, interrupts, inspectable topology, or a graph runtime that owns fan-out/fan-in scheduling.

Choose **Claude Agent SDK** when the workflow benefits from SDK-native agent sessions, Claude Code capabilities, MCP tools, permission callbacks, hooks, subagents, or resumable conversational work. For this benchmark, ordinary Python (`asyncio.gather`) owns the cross-session orchestration.

A real system can combine them: LangGraph can own durable business workflow state while a node invokes a narrowly scoped Claude Agent SDK session. That hybrid adds operational power, but also adds two tracing, retry, timeout, and state models—use it only when each layer has a clear responsibility.

### Discussion prompts

1. Which architecture makes the five-way join easier to inspect before execution?
2. Which exposes richer session-level evidence without adding a separate session layer?
3. Why can token totals differ even with identical user prompts?
4. Which timing components are framework overhead, provider latency, process startup, or output-length effects?
5. What would you add for a statistically defensible benchmark?

### Important limit

This notebook preloads authoritative evidence to isolate orchestration. It does not replace the grounded tool demonstrations in the Day 1 chains. In production, retrieval/tool calls, retries, rate limits, and checkpoint persistence can dominate both runtime and token use.